In [1]:
# %% [markdown]
# # CMAPSS + MLP + Feature Selection (Laplacian vs PCA vs KPCA)

# %% 
!pip install skfeature-chappers

# %%
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA, KernelPCA
from skfeature.function.similarity_based import lap_score
from skfeature.utility import construct_W

# If you keep step1 as a .py file:
from step1_ml_pipeline import load_cmapss_data, data_processing



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable


In [5]:
# %%
# ----- MLPRegressorCustom (exactly as you already have it) -----

import numpy as np

def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))
def sigmoid_prime(a): return a * (1.0 - a)

def tanh(x): return np.tanh(x)
def tanh_prime(a): return 1.0 - a**2

def relu(x): return np.maximum(0.0, x)
def relu_prime(a): return (a > 0.0).astype(float)

_ACTS = {
    "sigmoid": (sigmoid, sigmoid_prime),
    "tanh":    (tanh, tanh_prime),
    "relu":    (relu, relu_prime),
}

class MLPRegressorCustom:
    def __init__(
        self,
        size_hidden,
        activation="relu",
        weight_init="kaiming",
        learning_rate=1e-3,
        batch_size=256,
        l1=0.0,
        l2=1e-4,
        learning_rate_decay=0.0,
        loss_fn="huber",
        huber_delta=20.0,
        max_grad_norm=None,
        random_state=None,
        verbose=False,
    ):
        self.size_hidden = list(size_hidden)
        self.activation_name = activation
        self.activation, self.activation_prime = _ACTS[activation]
        self.weight_init = weight_init
        self.lr0 = learning_rate
        self.batch_size = int(batch_size)
        self.l1 = float(l1)
        self.l2 = float(l2)
        self.decay = float(learning_rate_decay)
        self.loss_fn = loss_fn
        self.delta = float(huber_delta)
        self.max_grad_norm = max_grad_norm
        self.rng = np.random.default_rng(seed=random_state)
        self.verbose = verbose

        self.W = []
        self.b = []
        self.history_ = {"train_loss": [], "val_loss": []}

    def _init_layer(self, in_dim, out_dim):
        if self.weight_init == "zeros":
            W = np.zeros((in_dim, out_dim))
        elif self.weight_init == "gaussian":
            W = self.rng.normal(0.0, 1.0, size=(in_dim, out_dim))
        elif self.weight_init == "uniform":
            W = self.rng.uniform(-1.0, 1.0, size=(in_dim, out_dim))
        elif self.weight_init == "xavier":
            bound = np.sqrt(6.0 / (in_dim + out_dim))
            W = self.rng.uniform(-bound, bound, size=(in_dim, out_dim))
        elif self.weight_init == "kaiming":
            std = np.sqrt(2.0 / in_dim)
            W = self.rng.normal(0.0, std, size=(in_dim, out_dim))
        else:
            raise ValueError(f"Unknown weight_init: {self.weight_init}")
        b = np.zeros((1, out_dim))
        return W, b

    def _init_params(self, d_in):
        layer_sizes = [d_in] + self.size_hidden + [1]
        self.W, self.b = [], []
        for i in range(len(layer_sizes) - 1):
            W, b = self._init_layer(layer_sizes[i], layer_sizes[i+1])
            self.W.append(W)
            self.b.append(b)

    def _forward(self, X):
        a = X
        acts = [a]
        zs = []
        for i in range(len(self.W) - 1):
            z = a @ self.W[i] + self.b[i]
            a = self.activation(z)
            zs.append(z)
            acts.append(a)
        z = acts[-1] @ self.W[-1] + self.b[-1]
        y_hat = z
        zs.append(z)
        acts.append(y_hat)
        return acts, zs

    def _loss_and_grad_last(self, y_hat, y_true):
        y_true = y_true.reshape(-1, 1)
        if self.loss_fn == "mse":
            loss = 0.5 * np.mean((y_hat - y_true) ** 2)
            dL_dy = (y_hat - y_true) / y_true.shape[0]
        elif self.loss_fn == "huber":
            e = y_hat - y_true
            abs_e = np.abs(e)
            quad = 0.5 * (e ** 2)
            lin  = self.delta * (abs_e - 0.5 * self.delta)
            loss = np.mean(np.where(abs_e <= self.delta, quad, lin))
            dL_dy = np.where(abs_e <= self.delta, e, self.delta * np.sign(e)) / y_true.shape[0]
        else:
            raise ValueError("loss_fn must be 'mse' or 'huber'")
        return loss, dL_dy

    def _regularize(self, grads_W):
        if self.l2 > 0.0:
            grads_W = [gW + self.l2 * W for gW, W in zip(grads_W, self.W)]
        if self.l1 > 0.0:
            grads_W = [gW + self.l1 * np.sign(W) for gW, W in zip(grads_W, self.W)]
        return grads_W

    def _clip(self, gW, gb):
        if self.max_grad_norm is None:
            return gW, gb
        total = 0.0
        for g in gW + gb:
            total += np.sum(g * g)
        total = np.sqrt(total)
        if total > self.max_grad_norm and total > 0.0:
            scale = self.max_grad_norm / total
            gW = [g * scale for g in gW]
            gb = [g * scale for g in gb]
        return gW, gb

    def _update(self, grads_W, grads_b, t):
        lr = self.lr0 / (1.0 + t * self.decay) if self.decay > 0.0 else self.lr0
        for i in range(len(self.W)):
            self.W[i] -= lr * grads_W[i]
            self.b[i] -= lr * grads_b[i]

    def _iterate_minibatches(self, X, y, batch_size, shuffle=True):
        n = X.shape[0]
        idx = np.arange(n)
        if shuffle:
            self.rng.shuffle(idx)
        for start in range(0, n, batch_size):
            end = min(start + batch_size, n)
            sel = idx[start:end]
            yield X[sel], y[sel]

    def fit(self, X_tr, y_tr, X_val=None, y_val=None, epochs=100, patience=10):
        y_tr = y_tr.reshape(-1, 1).astype(float)
        if X_val is not None:
            y_val = y_val.reshape(-1, 1).astype(float)

        self._init_params(X_tr.shape[1])
        best_loss = np.inf
        best_params = None
        no_improve = 0
        step = 0

        for epoch in range(1, epochs + 1):
            train_losses = []
            for xb, yb in self._iterate_minibatches(X_tr, y_tr, self.batch_size, shuffle=True):
                acts, zs = self._forward(xb)
                y_hat = acts[-1]

                loss, dL_dy = self._loss_and_grad_last(y_hat, yb)
                train_losses.append(loss)

                grads_W = [None] * len(self.W)
                grads_b = [None] * len(self.b)

                a_prev = acts[-2]
                grads_W[-1] = a_prev.T @ dL_dy
                grads_b[-1] = np.sum(dL_dy, axis=0, keepdims=True)

                delta = dL_dy @ self.W[-1].T
                for i in range(len(self.W) - 2, -1, -1):
                    d_act = self.activation_prime(acts[i+1])
                    delta *= d_act
                    grads_W[i] = acts[i].T @ delta
                    grads_b[i] = np.sum(delta, axis=0, keepdims=True)
                    if i > 0:
                        delta = delta @ self.W[i].T

                grads_W = self._regularize(grads_W)
                grads_W, grads_b = self._clip(grads_W, grads_b)
                self._update(grads_W, grads_b, t=step)
                step += 1

            tr_loss = float(np.mean(train_losses))
            self.history_["train_loss"].append(tr_loss)

            if X_val is not None:
                val_loss = float(self.loss(self.predict_raw(X_val), y_val))
                self.history_["val_loss"].append(val_loss)
                if self.verbose:
                    print(f"Epoch {epoch:3d} | train {tr_loss:.4f} | val {val_loss:.4f}")

                if val_loss + 1e-9 < best_loss:
                    best_loss = val_loss
                    best_params = ([W.copy() for W in self.W], [b.copy() for b in self.b])
                    no_improve = 0
                else:
                    no_improve += 1
                    if no_improve >= patience:
                        if self.verbose:
                            print(f"Early stopping at epoch {epoch} (best val {best_loss:.4f})")
                        break
            else:
                if self.verbose:
                    print(f"Epoch {epoch:3d} | train {tr_loss:.4f}")

        if best_params is not None:
            self.W, self.b = best_params

        return self

    def predict_raw(self, X):
        return self._forward(X)[0][-1]

    def predict(self, X):
        return self.predict_raw(X).ravel()

    def loss(self, y_pred, y_true):
        y_true = y_true.reshape(-1, 1)
        return self._loss_and_grad_last(y_pred, y_true)[0]


In [6]:
# %%
def get_nn_data(dataname="FD001", val_frac=0.15, shuffle_engines=True, random_state=42):
    # 1) load + scale using step1
    train_df, test_df = load_cmapss_data(dataname)
    train_std, test_std, scaler = data_processing(train_df, test_df)

    exclude = ["engine_id", "cycle", "RUL"]
    feature_cols = [c for c in train_std.columns if c not in exclude]

    X_train_full = train_std[feature_cols].to_numpy()
    y_train_full = train_std["RUL"].to_numpy()
    X_test = test_std[feature_cols].to_numpy()
    y_test = test_std["RUL"].to_numpy()

    eng_ids = train_std["engine_id"].unique()
    if shuffle_engines:
        rng = np.random.default_rng(seed=random_state)
        rng.shuffle(eng_ids)
    cut = int((1.0 - val_frac) * len(eng_ids))
    val_ids = set(eng_ids[cut:])
    mask_val = train_std["engine_id"].isin(val_ids).to_numpy()

    X_tr, y_tr = X_train_full[~mask_val], y_train_full[~mask_val]
    X_val, y_val = X_train_full[mask_val],  y_train_full[mask_val]

    meta = dict(feature_cols=feature_cols, scaler=scaler, val_engine_ids=sorted(list(val_ids)))
    return X_tr, y_tr, X_val, y_val, X_test, y_test, meta

# %%
X_tr, y_tr, X_val, y_val, X_test, y_test, meta = get_nn_data(
    dataname="FD001", val_frac=0.15, shuffle_engines=True, random_state=42
)
print(f"[info] features={len(meta['feature_cols'])}")
print(f"[shapes] X_tr={X_tr.shape}  X_val={X_val.shape}  X_test={X_test.shape}")


[info] features=21
[shapes] X_tr=(17567, 21)  X_val=(3064, 21)  X_test=(13096, 21)


In [7]:
# %% [markdown]
# ## Feature Selection: Laplacian Score & PCA Ranking

# %%
# X_tr is already standardized by data_processing (assumption).
X_tr_np = X_tr.copy()

# --- Laplacian Score (local structure, unsupervised) ---
print("[FS] computing Laplacian Score...")

W_adj = construct_W.construct_W(
    X_tr_np,
    neighbor_mode='knn',
    f=5,  # number of neighbors
    t=1   # heat kernel width
)

lap_scores = lap_score.lap_score(X_tr_np, W=W_adj)
ranked_lap = np.argsort(lap_scores)  # lower is better

print("Laplacian - top 10 feature indices:", ranked_lap[:10])
print("Laplacian - top 10 scores:", lap_scores[ranked_lap[:10]])


[FS] computing Laplacian Score...
Laplacian - top 10 feature indices: [ 6 11 20 19  4 16  8  1  9 10]
Laplacian - top 10 scores: [0 1 2 3 4 5 6 7 8 9]


In [8]:
# %%
# --- PCA-based feature importance (global variance) ---
print("[FS] computing PCA-based feature ranking...")

n_features = X_tr_np.shape[1]
pca = PCA(n_components=min(10, n_features))
pca.fit(X_tr_np)

# Sum absolute loadings across components
feat_importance = np.abs(pca.components_).sum(axis=0)
ranked_pca = np.argsort(-feat_importance)  # higher importance first

print("PCA - top 10 feature indices:", ranked_pca[:10])
print("PCA - top 10 importances:", feat_importance[ranked_pca[:10]])


[FS] computing PCA-based feature ranking...
PCA - top 10 feature indices: [19 14  6  1 20  3  7 16  2  8]
PCA - top 10 importances: [2.38287924 2.26924774 2.00854402 2.00121888 1.98124654 1.98052377
 1.9193985  1.8453221  1.72616319 1.15205498]


In [9]:
# %%
def evaluate_model_on_features(method_name, feature_order, d_list, 
                               X_tr, y_tr, X_val, y_val, X_test, y_test,
                               epochs=250, patience=25, verbose=False):
    results = []

    for d in d_list:
        d = min(d, X_tr.shape[1])  # safety
        cols = feature_order[:d]

        X_tr_fs = X_tr[:, cols]
        X_val_fs = X_val[:, cols]
        X_test_fs = X_test[:, cols]

        if verbose:
            print(f"\n[{method_name}] d={d} features")

        model = MLPRegressorCustom(
            size_hidden=[256, 128, 64],
            activation="relu",
            weight_init="kaiming",
            learning_rate=1e-3,
            batch_size=256,
            l1=0.0, l2=1e-4,
            learning_rate_decay=0.0,
            loss_fn="huber", huber_delta=20.0,
            max_grad_norm=5.0,
            random_state=42,
            verbose=verbose,
        )

        model.fit(X_tr_fs, y_tr, X_val_fs, y_val, epochs=epochs, patience=patience)

        y_pred = model.predict(X_test_fs)

        mae  = float(np.mean(np.abs(y_test - y_pred)))
        rmse = float(np.sqrt(np.mean((y_test - y_pred)**2)))
        ss_res = float(np.sum((y_test - y_pred)**2))
        ss_tot = float(np.sum((y_test - np.mean(y_test))**2))
        r2 = 1.0 - ss_res/ss_tot

        results.append({
            "method": method_name,
            "d_features": d,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2,
        })

    return pd.DataFrame(results)


In [10]:
# %%
# choose some candidate numbers of features (adjust based on actual dimension)
n_features = X_tr.shape[1]
d_list = [5, 10, 15, 20, n_features]

df_lap = evaluate_model_on_features(
    "Laplacian", ranked_lap, d_list,
    X_tr, y_tr, X_val, y_val, X_test, y_test,
    epochs=250, patience=25, verbose=False
)

df_pca = evaluate_model_on_features(
    "PCA", ranked_pca, d_list,
    X_tr, y_tr, X_val, y_val, X_test, y_test,
    epochs=250, patience=25, verbose=False
)

# Baseline: all features without FS
df_all = evaluate_model_on_features(
    "AllFeatures", np.arange(n_features), [n_features],
    X_tr, y_tr, X_val, y_val, X_test, y_test,
    epochs=250, patience=25, verbose=False
)

results_fs = pd.concat([df_lap, df_pca, df_all], ignore_index=True)
results_fs.sort_values(by="RMSE", inplace=True)
results_fs


,method,d_features,MAE,RMSE,R2
9,PCA,21,33.796403,45.875129,0.394971
4,Laplacian,21,33.811356,45.914656,0.393927
3,Laplacian,20,33.783522,45.917166,0.393861
8,PCA,20,33.836069,45.922928,0.393709
7,PCA,15,33.892050,45.950392,0.392984
10,AllFeatures,21,33.906528,45.962718,0.392658
2,Laplacian,15,34.527174,46.735782,0.372056
1,Laplacian,10,34.610862,46.772340,0.371073
6,PCA,10,34.631479,46.895167,0.367766
5,PCA,5,40.181213,52.984748,0.192907


In [11]:
# %% [markdown]
# ## KPCA Baseline (feature extraction)

# %%
print("[main] applying KPCA (RBF kernel)...", flush=True)
gamma = 1.0 / X_tr.shape[1]

kpca = KernelPCA(
    n_components=50,
    kernel="rbf",
    gamma=gamma,
    fit_inverse_transform=False,
    eigen_solver="auto",
    n_jobs=-1
)

X_tr_kpca  = kpca.fit_transform(X_tr)
X_val_kpca = kpca.transform(X_val)
X_test_kpca= kpca.transform(X_test)

print(f"[main] KPCA reduced features {X_tr.shape[1]} → {X_tr_kpca.shape[1]}", flush=True)

model_kpca = MLPRegressorCustom(
    size_hidden=[256, 128, 64],
    activation="relu",
    weight_init="kaiming",
    learning_rate=1e-3,
    batch_size=256,
    l1=0.0, l2=1e-4,
    learning_rate_decay=0.0,
    loss_fn="huber", huber_delta=20.0,
    max_grad_norm=5.0,
    random_state=42,
    verbose=True,
)

model_kpca.fit(X_tr_kpca, y_tr, X_val_kpca, y_val, epochs=250, patience=25)
y_pred_kpca = model_kpca.predict(X_test_kpca)

mae_kpca  = float(np.mean(np.abs(y_test - y_pred_kpca)))
rmse_kpca = float(np.sqrt(np.mean((y_test - y_pred_kpca)**2)))
ss_res    = float(np.sum((y_test - y_pred_kpca)**2))
ss_tot    = float(np.sum((y_test - np.mean(y_test))**2))
r2_kpca   = 1.0 - ss_res/ss_tot

print(f"[KPCA] MAE={mae_kpca:.2f}  RMSE={rmse_kpca:.2f}  R2={r2_kpca:.3f}")


[main] applying KPCA (RBF kernel)...
[main] KPCA reduced features 21 → 50
Epoch   1 | train 1961.5870 | val 1912.0472
Epoch   2 | train 1947.8089 | val 1895.4606
Epoch   3 | train 1927.9147 | val 1872.1997
Epoch   4 | train 1901.3765 | val 1840.9961
Epoch   5 | train 1864.4799 | val 1799.7900
Epoch   6 | train 1817.5539 | val 1746.5447
Epoch   7 | train 1757.0896 | val 1679.3816
Epoch   8 | train 1683.4108 | val 1596.9197
Epoch   9 | train 1591.7482 | val 1498.6107
Epoch  10 | train 1485.6600 | val 1384.7760
Epoch  11 | train 1363.9647 | val 1257.4166
Epoch  12 | train 1229.5148 | val 1121.3953
Epoch  13 | train 1090.8937 | val 985.7878
Epoch  14 | train 959.7041 | val 864.8600
Epoch  15 | train 849.1740 | val 772.2906
Epoch  16 | train 769.0822 | val 705.0196
Epoch  17 | train 705.8256 | val 646.8000
Epoch  18 | train 648.9714 | val 592.0537
Epoch  19 | train 595.8955 | val 542.8372
Epoch  20 | train 549.1767 | val 502.3064
Epoch  21 | train 512.2674 | val 472.9458
Epoch  22 | train 4